# Paper Table 1 evaluation

This notebook produces the per-stage evaluation table used in Sec. 9 of the ISMIR paper *"Raw Note Transcription for Hardanger Fiddle via a Hybrid Neural/Rule-Based Approach"*.

For each tune (Spretten, Godvaersdagen) and each post-processing stage (raw Transkun MIDI, +pitch refinement CSV, +offset synchronisation CSV) it compares against the performer-annotated ground truth (`*_truther*.csv`) and computes:

- **Standard F1** — onset tol 50 ms, offset tol max(50 ms, 20% dur), pitch tol 50 cents (MIREX/MAESTRO defaults: see `references.bib`).
- **Strict F1** — offset tol max(25 ms, 5% dur), same other tolerances. Sensitivity variant; not a separately named metric.
- **Deviation MAE** on matched notes — onset MAE [ms], offset MAE [ms], pitch MAE [cents].

Outputs:
- `table_results.csv` — full table (per-tune rows + aggregate).
- `table_results.tex` — LaTeX `tabular` ready to drop into the paper.

In [1]:
from pathlib import Path
import sys, glob
import numpy as np
import pandas as pd

# eval_utils.py lives next to this notebook
sys.path.insert(0, str(Path.cwd()))
import eval_utils as ev

POSTPROS = Path('postpros')
TUNES = ['Spretten', 'Godvaersdagen']
STAGES = ['raw', '+pitch', '+offset']

In [2]:
# Auto-discover tunes from postpros/. Adding a new tune is just dropping
# its truther + raw .mid + pitch/offset CSVs in the directory.
entries = ev.discover(POSTPROS)
print(f'Discovered {len(entries)} tune(s):')
for e in entries:
    print(f"  {e['tune']}")
    print(f"    truth:  {Path(e['truth']).name}")
    for k, v in e['stages'].items():
        print(f"    {k:<8} {Path(v).name if v else '(missing)'}")


Discovered 2 tune(s):
  Godvaersdagen
    truth:  Godvaersdagen_truther 27-Apr-2026 21-06-04.csv
    raw      Godvaersdagen_original1_transkun239.mid
    +pitch   Godvaersdagen_original1_transkun239_pitch 27-Apr-2026 21-04-05.csv
    +offset  Godvaersdagen_original1_transkun239_offset 27-Apr-2026 21-04-39.csv
  Spretten
    truth:  Spretten_truther 27-Apr-2026 21-01-35.csv
    raw      Spretten_original_transkun239.mid
    +pitch   Spretten_original_transkun239_pitch 27-Apr-2026 20-54-25.csv
    +offset  Spretten_original_transkun239_offset 27-Apr-2026 20-56-16.csv


In [3]:
# Surface silently bundled transformations and identical-stage warnings.
for e in entries:
    p_pitch = e['stages']['+pitch']
    p_offset = e['stages']['+offset']
    if p_pitch and p_offset and ev.diagnose_identical(p_pitch, p_offset):
        print(f"WARNING: {e['tune']}: +pitch and +offset CSVs are byte-identical on "
              f"(onset, offset, onpitch). The +offset row will repeat the +pitch row.")


In [4]:
rows = []
for e in entries:
    truth = e['truth']
    for stage in STAGES:
        est = e['stages'][stage]
        if est is None:
            continue
        r = ev.evaluate_pair(truth, est)
        rows.append({'tune': e['tune'], 'stage': stage, **r})
        print(f"{e['tune']:18s} {stage:8s} F_std={r['F_std']:.3f}  F_strict={r['F_strict']:.3f}  "
              f"onset_mae={r['onset_mae_ms']:5.2f}ms  offset_mae={r['offset_mae_ms']:5.2f}ms  "
              f"pitch_mae={r['pitch_mae_cents']:5.2f}ct  n_match={r['n_match_std']}/{r['n_match_offset']}")

per_pair = pd.DataFrame(rows)
per_pair[['tune', 'stage', 'n_ref', 'F_std', 'F_strict',
          'onset_mae_ms', 'offset_mae_ms', 'pitch_mae_cents',
          'n_match_std', 'n_match_offset']].round(3)


Godvaersdagen      raw      F_std=0.902  F_strict=0.835  onset_mae= 6.85ms  offset_mae=10.76ms  pitch_mae=10.99ct  n_match=1093/1033
Godvaersdagen      +pitch   F_std=0.843  F_strict=0.698  onset_mae= 6.93ms  offset_mae=13.91ms  pitch_mae=11.98ct  n_match=1055/962
Godvaersdagen      +offset  F_std=0.843  F_strict=0.698  onset_mae= 6.93ms  offset_mae=13.91ms  pitch_mae=11.98ct  n_match=1055/962
Spretten           raw      F_std=0.860  F_strict=0.787  onset_mae= 7.61ms  offset_mae=11.12ms  pitch_mae= 8.56ct  n_match=666/610
Spretten           +pitch   F_std=0.798  F_strict=0.620  onset_mae= 7.51ms  offset_mae=15.85ms  pitch_mae=12.54ct  n_match=630/560
Spretten           +offset  F_std=0.798  F_strict=0.620  onset_mae= 7.51ms  offset_mae=15.85ms  pitch_mae=12.54ct  n_match=630/560


,tune,stage,n_ref,F_std,F_strict,onset_mae_ms,offset_mae_ms,pitch_mae_cents,n_match_std,n_match_offset
0,Godvaersdagen,raw,1126,0.902,0.835,6.848,10.760,10.988,1093,1033
1,Godvaersdagen,+pitch,1126,0.843,0.698,6.932,13.908,11.979,1055,962
2,Godvaersdagen,+offset,1126,0.843,0.698,6.932,13.908,11.979,1055,962
3,Spretten,raw,726,0.860,0.787,7.606,11.120,8.562,666,610
4,Spretten,+pitch,726,0.798,0.620,7.508,15.854,12.541,630,560
5,Spretten,+offset,726,0.798,0.620,7.508,15.854,12.541,630,560


In [5]:
# Note-weighted aggregate over the two tunes (MIREX micro-averaging convention)
def weighted_mean(df, value_col, weight_col='n_ref'):
    w = df[weight_col]
    return float((df[value_col] * w).sum() / w.sum())

agg_rows = []
for stage in STAGES:
    sub = per_pair[per_pair['stage'] == stage]
    agg_rows.append({
        'tune': 'AGGREGATE',
        'stage': stage,
        'n_ref': int(sub['n_ref'].sum()),
        'n_est': int(sub['n_est'].sum()),
        'P_std': weighted_mean(sub, 'P_std'),
        'R_std': weighted_mean(sub, 'R_std'),
        'F_std': weighted_mean(sub, 'F_std'),
        'P_strict': weighted_mean(sub, 'P_strict'),
        'R_strict': weighted_mean(sub, 'R_strict'),
        'F_strict': weighted_mean(sub, 'F_strict'),
        'onset_mae_ms': weighted_mean(sub, 'onset_mae_ms'),
        'offset_mae_ms': weighted_mean(sub, 'offset_mae_ms'),
        'pitch_mae_cents': weighted_mean(sub, 'pitch_mae_cents'),
        'n_match_std': int(sub['n_match_std'].sum()),
        'n_match_offset': int(sub['n_match_offset'].sum()),
    })

agg = pd.DataFrame(agg_rows)
agg

,tune,stage,n_ref,n_est,P_std,R_std,F_std,P_strict,R_strict,F_strict,onset_mae_ms,offset_mae_ms,pitch_mae_cents,n_match_std,n_match_offset
0,AGGREGATE,raw,1852,1857,0.884660,0.887149,0.885551,0.815540,0.818035,0.816462,7.145001,10.901413,10.036856,1759,1643
1,AGGREGATE,+pitch,1852,1835,0.829304,0.821814,0.825099,0.670325,0.665227,0.667413,7.158015,14.670818,12.199055,1685,1522
2,AGGREGATE,+offset,1852,1835,0.829304,0.821814,0.825099,0.670325,0.665227,0.667413,7.158015,14.670818,12.199055,1685,1522


In [6]:
# Diagnostics: pitch bias, duration floor, identical-stage detection per (tune, stage).
diag_rows = []
for e in entries:
    for stage, p in e['stages'].items():
        if p is None:
            continue
        d = ev.diagnose_stage(e['truth'], p)
        diag_rows.append({'tune': e['tune'], 'stage': stage, **d})
diag = pd.DataFrame(diag_rows)
diag[['tune', 'stage', 'n_est', 'pitch_bias_cents', 'pitch_p95_cents',
      'duration_floor_ms', 'duration_floor_count']].round(2)


,tune,stage,n_est,pitch_bias_cents,pitch_p95_cents,duration_floor_ms,duration_floor_count
0,Godvaersdagen,raw,1165,10.01,35.50,1.04,1
1,Godvaersdagen,+pitch,1157,-10.10,42.00,99.99,245
2,Godvaersdagen,+offset,1157,-10.10,42.00,99.99,245
3,Spretten,raw,692,-1.54,26.00,3.65,2
4,Spretten,+pitch,678,-11.24,56.45,100.00,153
5,Spretten,+offset,678,-11.24,56.45,100.00,153


In [7]:
# Sanity: strict F1 cannot exceed standard F1 (stricter offset constraint <=> fewer matches)
for _, row in per_pair.iterrows():
    assert row['F_strict'] <= row['F_std'] + 1e-9, \
        f"Strict > Standard at {row['tune']}/{row['stage']}"
print('All (tune, stage) rows: F_strict <= F_std OK')

All (tune, stage) rows: F_strict <= F_std OK


In [8]:
out_csv = Path('table_results.csv')
combined = pd.concat([per_pair, agg], ignore_index=True)
combined.to_csv(out_csv, index=False, float_format='%.4f')
print(f'Wrote {out_csv.resolve()}')

# LaTeX table for paper Sec 9 (aggregate row only - matches the paper's tabular spec)
stage_labels = {
    'raw': 'Raw model',
    '+pitch': '+ pitch refinement',
    '+offset': '+ offset synchronisation',
}

EOL = chr(92) + chr(92)  # two literal backslashes for end-of-row

def latex_row(stage_label, row):
    return (f"{stage_label} & {row['F_std']*100:.2f} & {row['F_strict']*100:.2f} & "
            f"{row['onset_mae_ms']:.1f} & {row['offset_mae_ms']:.1f} & {row['pitch_mae_cents']:.1f} {EOL}")

latex = []
latex.append(r'\begin{tabular}{lccccc}')
latex.append(r'\hline')
latex.append(f'Stage & F1 & F1 & Onset & Offset & Pitch {EOL}')
latex.append(f'      & (std) & (strict) & MAE [ms] & MAE [ms] & MAE [ct] {EOL}')
latex.append(r'\hline')
for stage in STAGES:
    row = agg[agg['stage'] == stage].iloc[0]
    latex.append(latex_row(stage_labels[stage], row))
latex.append(r'\hline')
latex.append(r'\end{tabular}')

out_tex = Path('table_results.tex')
out_tex.write_text('\n'.join(latex) + '\n')
print(f'Wrote {out_tex.resolve()}')
print()
print('\n'.join(latex))


Wrote /Users/lrs1/Documents/october2025/paper27/AMT-evaluation/table_results.csv
Wrote /Users/lrs1/Documents/october2025/paper27/AMT-evaluation/table_results.tex

\begin{tabular}{lccccc}
\hline
Stage & F1 & F1 & Onset & Offset & Pitch \\
      & (std) & (strict) & MAE [ms] & MAE [ms] & MAE [ct] \\
\hline
Raw model & 88.56 & 81.65 & 7.1 & 10.9 & 10.0 \\
+ pitch refinement & 82.51 & 66.74 & 7.2 & 14.7 & 12.2 \\
+ offset synchronisation & 82.51 & 66.74 & 7.2 & 14.7 & 12.2 \\
\hline
\end{tabular}


## Caveats / findings (from the diagnostics cell above)

The current postpros export bundles three transformations into the **+pitch** stage. Read F1/MAE deltas with this in mind:

1. **The +pitch stage applies a 100 ms minimum-duration floor.** 22.5% of Spretten notes (153/678) and 21.2% of Godvaersdagen notes (245/1157) end up at exactly 100.0 ms. This shifts offsets by mean +8 ms / std 17 ms and pushes ~18% of notes past the strict tolerance — that drop alone explains most of the strict-F1 collapse, not the pitch refinement.
2. **The pitch refinement has a systematic ~10-cent downward bias.** Raw pitch error vs truth: −1.5 c / +10.0 c (Spretten/Godvaersdagen). After +pitch: −11.2 c / −10.1 c — both flat by about a dime. About a dozen Spretten notes shift past the 50-cent matching tolerance entirely, breaking matches that the raw integer pitches accidentally got right. Likely cause: post-processor uses a slightly different reference frequency than the truth annotation, or the harmonic-sieve inhibition pulls fundamentals down. Worth investigating.
3. **+pitch and +offset CSVs are byte-identical on (onset, offset, onpitch).** They differ only in the `previous`/`next` voice-chain pointers. The +offset row of Table 1 will repeat the +pitch row until the offset-stage post-processor is re-exported with actual offset corrections (or the third stage is renamed to "+ voice segregation").
4. **MAE is conditional on a match.** Read `n_match_std` (used for onset/pitch MAE) and `n_match_offset` (used for offset MAE) alongside the deviation values — a stage with lower MAE but a lower match count is not necessarily better.
5. **Raw pitch MAE has a quantization floor.** The raw stage reports integer MIDI pitches; the truth has fractional pitches. Pitch MAE on raw is bounded below by the cents-deviation of truth from the nearest semitone — for Hardanger fiddle (non-equal temperament + scordatura) this floor is non-trivial. Part of any raw → +pitch *change* is dequantization, not pure model improvement.